# PersonaPlex — RunPod RTX 5090 Deployment Notebook

## 1. Environment sanity checks

Confirms this is a Linux pod with a GPU attached and a supported Python version (`moshi-personaplex` requires
Python ≥ 3.10, per `moshi/pyproject.toml`).

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")

In [ ]:
# Confirm the NVIDIA driver sees a GPU at the OS level before we install anything.
!nvidia-smi

## 2. Persistent storage setup (RunPod volume)

In [ ]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-RAG-code-streaming-v3.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"   # loaders.DEFAULT_REPO in moshi/moshi/models/loaders.py

SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 (not "localhost") so RunPod's proxy can reach the server
SERVER_PORT = 8998        # default port used by moshi.server

# RunPod's HTTP port proxy terminates TLS at its edge and forwards plain HTTP to the container, so by
# default we do NOT enable the app's own self-signed TLS (--ssl). Flip this to True only if you plan to
# expose SERVER_PORT directly as a raw TCP port instead of through RunPod's HTTP proxy.
USE_APP_TLS = False

# An RTX 5090 has 32GB of VRAM, comfortably enough for this 7B model in bf16 -- CPU offload should not be
# needed. Flip to True only if you hit CUDA OOM (requires the `accelerate` package, installed below anyway).
USE_CPU_OFFLOAD = False

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
# Keep PATH aware of ~/.local/bin, where moshi/moshi/utils/connection.py installs `mkcert` if --ssl is used.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("WORKSPACE   :", WORKSPACE)
print("REPO_DIR    :", REPO_DIR)
print("HF_HOME     :", os.environ["HF_HOME"])
print("USE_APP_TLS :", USE_APP_TLS)

## 3. System package installation

Per the repo's `README.md` prerequisites: install the **Opus codec development library** before installing the
Python package (it's required by `sphn`, which PersonaPlex uses for Opus audio streaming over the WebSocket).
We also make sure `git` is present for cloning.

In [ ]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "

!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")

## 4. Repository cloning

If `REPO_DIR` doesn't already contain the project (e.g. you uploaded your own copy of this repo to the volume
beforehand), it is cloned fresh from GitHub. If it's already there, cloning is skipped automatically — this
cell is safe to re-run on every pod restart.

In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"

if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."


## 5. Python dependency installation

The repo's documented install command is:

```bash
pip install moshi/.
```

which installs from `moshi/pyproject.toml` (`numpy`, `safetensors`, `huggingface-hub`, `einops`,
`sentencepiece`, `sounddevice`, `sphn`, `torch>=2.2,<2.5`, `aiohttp`).

### RTX 5090 / Blackwell note

The pinned `torch<2.5` build does **not** ship CUDA kernels for Blackwell (RTX 50‑series, `sm_120`) GPUs. The
repo's `README.md` explicitly documents the fix for this
([NVIDIA/personaplex#2](https://github.com/NVIDIA/personaplex/issues/2)): reinstall PyTorch from the `cu130`
wheel index *after* the base install. This intentionally overrides the `<2.5` pin — that's expected and is the
upstream-recommended fix, not a mistake.

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/." 

In [ ]:
# Blackwell (RTX 5090) requires CUDA-13.0-built PyTorch wheels. This intentionally supersedes the
# torch<2.5 pin from moshi/pyproject.toml -- see README.md "Extra step for Blackwell based GPUs".
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# `accelerate` enables --cpu-offload (README's "CPU Offload" section); `gradio` enables the optional
# --gradio-tunnel fallback for public exposure if you ever need it instead of RunPod's HTTP proxy.
%pip install -q accelerate gradio

## 6. CUDA / GPU verification

Confirms PyTorch can see the RTX 5090 and that the installed build actually has working CUDA kernels for it
(a bf16 matmul smoke test) — this is the check that would have failed before the `cu130` reinstall above if it
had been skipped.

In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by PyTorch. Confirm this RunPod pod has an RTX 5090 GPU attached "
        "and that the driver is healthy (see the `nvidia-smi` output above)."
    )

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU                :", device_name)
print("Compute capability :", capability)

if "5090" not in device_name and "RTX 50" not in device_name:
    print(f"WARNING: expected an RTX 5090, found '{device_name}'. Continuing anyway.")

# Smoke test: this is exactly the kind of op that fails with
# "no kernel image is available for execution on the device" on Blackwell if torch wasn't
# reinstalled from the cu130 index.
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
print("bf16 CUDA matmul smoke test OK, result shape:", tuple(y.shape))

## 7. Hugging Face authentication

**Manual step required before running this cell:** log into Hugging Face in a browser, open
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), and click **"Agree and access
repository"** to accept the NVIDIA Open Model License. Then create an access token (read access is enough) at
<https://huggingface.co/settings/tokens>.

The repo's `README.md` documents this as `export HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>`; the cell below does the
same thing from inside the notebook, via a hidden prompt so the token isn't echoed into cell output.

In [ ]:
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt' 
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## 8. Model downloading

`moshi/moshi/server.py` and `moshi/moshi/offline.py` both lazily call `hf_hub_download` for each asset the
first time they need it. We pre-fetch the same files here so that (a) any license/token problem surfaces now
with a clear error instead of mid-startup, and (b) everything is already warm in the `HF_HOME` cache before we
launch the server.

Assets (from `moshi/moshi/models/loaders.py` and `moshi/moshi/server.py`):
- `config.json` — model config
- `tokenizer_spm_32k_3.model` — SentencePiece text tokenizer
- `tokenizer-e351c8d8-checkpoint125.safetensors` — Mimi codec weights
- `model.safetensors` — Moshi/PersonaPlex LM weights
- `voices.tgz` — packaged voice-prompt embeddings (NATF0‑3, NATM0‑3, VARF0‑4, VARM0‑4)
- `dist.tgz` — the **prebuilt web UI** (this is why no separate frontend build step is needed)

In [ ]:
import tarfile

from huggingface_hub import hf_hub_download

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        # No explicit cache_dir: this honors HF_HOME (set in step 2) so the notebook and the server
        # subprocess we launch later (which inherits the same env) share one cache.
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )

In [ ]:
import pathlib

# Pre-extract the tarballs once, exactly like _get_voice_prompt_dir / _get_static_path do in
# moshi/moshi/server.py, so the first real request doesn't pay the extraction cost.
for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if out_dir.exists() else 'extracted now'})")

## 9. TLS certificate directory (only used if `USE_APP_TLS = True`)

The README's default local-machine workflow is:

```bash
SSL_DIR=$(mktemp -d); python -m moshi.server --ssl "$SSL_DIR"
```

`--ssl` makes `moshi/moshi/utils/connection.py` auto-install `mkcert` and generate a **self-signed**
certificate in that directory. That's appropriate for direct LAN/local access, but on RunPod we're putting the
server behind RunPod's own HTTPS proxy (Section 11), which already terminates TLS at the edge — adding a second,
self-signed TLS layer underneath it would just produce certificate warnings for no benefit. We still create the
directory here so you can flip `USE_APP_TLS = True` above if you'd rather expose `SERVER_PORT` as a raw TCP port
instead of through the HTTP proxy.

In [ ]:
import tempfile

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")
print("SSL_DIR:", SSL_DIR, "(only used if USE_APP_TLS = True)")

## 10. RAG knowledge-base configuration

This notebook grounds every conversation in a knowledge base loaded from a plain text file
(`rag/data/text.txt` by default -- replace it with your own company's data and re-run from here
down). RAG reuses PersonaPlex's own persona/voice-prompt injection mechanism (`rag/`), so no
changes to the model or the web UI are needed.

The assistant is restricted to answering ONLY from this knowledge base (`STRICT_SCOPE = True`
below): any question it isn't covered by gets an explicit decline (`REFUSAL_MESSAGE`) instead of
an answer made up from the model's own pretrained knowledge.

**Edit the variables in the next cell, then run every cell from here to the end of the notebook --
no other manual changes are required to get a working RAG-grounded streaming server.**

See `docs/PRODUCTION_RAG.md` for the full design writeup, including the root-cause analysis of why
naive character-offset chunking and an uncapped "inject everything" fallback silently broke once a
knowledge base grew large enough to approach the model's attention context window.

In [ ]:
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ---- Knowledge base -----------------------------------------------------------------------
RAG_TEXT_KB_PATH = os.path.join(REPO_DIR, "rag", "data", "text.txt")  # swap in your own file here
RAG_INDEX_PATH = os.path.join(WORKSPACE, "rag_indexes", "production")

# Chunking (rag.build_index.chunk_text): each blank-line-separated paragraph becomes one chunk by
# default -- paragraphs are never merged together (that dilutes their embedding and can drop a
# fact out of the retrieved set entirely; see docs/PRODUCTION_RAG.md). CHUNK_SIZE_CHARS only
# matters for paragraphs longer than this: they're sub-split on sentence, then word, then
# character boundaries -- never mid-word. MIN_CHUNK_CHARS is the one case paragraphs still get
# merged (short headers / "Field: value" fragments with no standalone retrieval signal).
CHUNK_SIZE_CHARS = 1000
OVERLAP_CHARS = 150
MIN_CHUNK_CHARS = 200

# ---- Retrieval + injection -----------------------------------------------------------------
ENABLE_RAG = True
INJECTION_MODE = "persona_rag"  # Mode C: the same <system> mechanism the persona prompt uses.
EMBEDDING_MODEL = "bge-small"   # bge-small | bge-base | bge-large | e5-small | e5-base | e5-large | sentence-transformers
VECTOR_DB = "faiss"

# TOP_K is deliberately generous: it bounds how many candidate chunks the similarity search
# ranks at all, NOT how many actually get injected -- that's the token-budget guard below. Set
# this >= your knowledge base's total chunk count so ranking always considers every chunk; the
# real safety cap is MAX_INJECTION_TOKENS / INJECTION_RESERVE_FRAMES.
TOP_K = 50

# The browser/voice web UI has no ASR, so it can never supply its own rag_query -- every real
# connection through it falls back to this fixed description for connection-start retrieval
# ranking. Keep it a short, broad summary of what the knowledge base covers.
RAG_DEFAULT_QUERY = (
    "RobotBulls company overview, products, automated trading robots, RBT token, "
    "research, and policies"
)

# Optional hard similarity cutoff for the explicit-query retrieval path (moshi.offline
# --rag-query / a client-supplied rag_query) -- None (the default) applies no cutoff at all,
# relying entirely on STRICT_SCOPE's instruction wording below to make the model decline.
# Measured empirically against this exact knowledge base with bge-small: clearly off-topic
# questions ("What's the capital of France?") scored ~0.42-0.56, while genuine but generically
# phrased in-scope questions ("How can I contact support?") scored as low as ~0.51 -- the two
# ranges OVERLAP, so any cutoff aggressive enough to reliably block off-topic questions will also
# occasionally false-decline a real, in-scope one. Only set this (e.g. 0.55-0.6) if you've
# measured your own knowledge base's score distribution and have accepted that tradeoff; leaving
# it None and relying on STRICT_SCOPE is the safer default.
SCORE_THRESHOLD = None

# ---- Scope enforcement: answer ONLY from the knowledge base, decline everything else --------
# Without this, injecting retrieved facts only ever *adds* context -- it never tells the model
# NOT to fall back on its own pretrained knowledge for whatever the knowledge base doesn't cover.
# STRICT_SCOPE=True wraps every injected knowledge block with an explicit "answer ONLY from this"
# instruction, and -- whenever retrieval finds nothing relevant at all (an explicit query scoring
# below SCORE_THRESHOLD, or an empty index) -- injects an explicit decline instruction instead of
# silently injecting nothing. See docs/PRODUCTION_RAG.md.
STRICT_SCOPE = True
REFUSAL_MESSAGE = "I can only answer questions based on the ROBOTBULLS Private Company documentation."

# ---- Token-budget guard (see docs/PRODUCTION_RAG.md) ----------------------------------------
# The live model's attention window (RingKVCache) has a FIXED capacity, shared by the persona
# prompt, the voice prompt, the injected RAG knowledge, and the live conversation that follows --
# it is a ring buffer, so injecting more tokens than it can hold silently evicts the OLDEST ones
# (the persona prompt and the earliest knowledge chunks) before the user has even spoken. Leave
# MAX_INJECTION_TOKENS unset (None) to compute the safe budget live from the connection's actual
# cache headroom every time -- the right choice for almost every deployment, including this one.
MAX_INJECTION_TOKENS = None
# Keep this SMALL: it directly trades off against how much of the knowledge base actually fits
# the injection budget. An earlier default of 400 frames (~32s) left too little room for this
# project's own ~12K-character RobotBulls text.txt (21 chunks, ~2,400-2,550 tokens) to fit
# entirely -- the lowest-ranked topics (against RAG_DEFAULT_QUERY) were silently dropped, making
# the assistant unable to answer questions about exactly those topics even though they ARE in the
# document. See docs/PRODUCTION_RAG.md Section 12. Only raise this once you've confirmed (Section
# 12's verification cell, or the server's rag_logs) that your knowledge base already fits
# comfortably and you specifically need headroom for very long calls.
INJECTION_RESERVE_FRAMES = 100  # frames reserved for the live conversation after injection (~8s @ 12.5Hz)

# Advanced -- only consulted if you change INJECTION_MODE above.
VAD_ENABLED = False
DYNAMIC_INJECTION_INTERVAL_S = 30.0
DYNAMIC_INJECTION_TOP_K = 2
TURN_INJECTION_TOP_K = 2
BENCHMARK_MODE = False

from rag.config import InjectionMode, RAGConfig

rag_config = RAGConfig(
    enable_rag=ENABLE_RAG,
    injection_mode=InjectionMode(INJECTION_MODE),
    top_k=TOP_K,
    embedding_model=EMBEDDING_MODEL,
    vector_db=VECTOR_DB,
    benchmark_mode=BENCHMARK_MODE,
    vad_enabled=VAD_ENABLED,
    dynamic_injection_interval_s=DYNAMIC_INJECTION_INTERVAL_S,
    dynamic_injection_top_k=DYNAMIC_INJECTION_TOP_K,
    turn_injection_top_k=TURN_INJECTION_TOP_K,
    default_query=RAG_DEFAULT_QUERY,
    score_threshold=SCORE_THRESHOLD,
    strict_scope=STRICT_SCOPE,
    refusal_message=REFUSAL_MESSAGE,
    max_injection_tokens=MAX_INJECTION_TOKENS,
    injection_reserve_frames=INJECTION_RESERVE_FRAMES,
    log_dir=os.path.join(WORKSPACE, "rag_logs"),
)

print(rag_config.describe())

### Install RAG dependencies and self-test the `rag/` package

Runs the unit tests that ship with `rag/` (see `rag/tests/`). These use plain Python stand-ins for
`LMGen` and the tokenizer -- they do **not** require the GPU, CUDA, or the PersonaPlex weights, so
this is safe to run at any point and catches a broken `rag/` install before any GPU time is spent.

In [ ]:
if ENABLE_RAG:
    %pip install -q faiss-cpu sentence-transformers
    print("RAG retrieval dependencies installed.")
else:
    print("ENABLE_RAG is False -- skipping faiss-cpu/sentence-transformers install.")

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "rag/tests", "-t", "."],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("rag/ unit tests failed -- see output above.")
print("rag/ scaffolding self-test passed.")

## 11. Build the FAISS index from the knowledge base

Chunks `RAG_TEXT_KB_PATH` with the parameters from Section 10 and embeds + indexes every chunk.
Prints a preview of every chunk so you can confirm the document split the way you expect (e.g. one
chunk per "Product X does Y" paragraph) before spending any GPU time.

In [ ]:
if not ENABLE_RAG:
    raise RuntimeError("Set ENABLE_RAG = True in Section 10 and re-run the cells above before running this one.")

import json

from rag.build_index import build_index_from_text_file, chunk_text

with open(RAG_TEXT_KB_PATH, encoding="utf-8") as f:
    _kb_text = f.read()
print(f"Knowledge source: {RAG_TEXT_KB_PATH} ({len(_kb_text)} chars)")

_preview_chunks = chunk_text(
    _kb_text, chunk_size_chars=CHUNK_SIZE_CHARS, overlap_chars=OVERLAP_CHARS, min_chunk_chars=MIN_CHUNK_CHARS
)
print(
    f"Chunked into {len(_preview_chunks)} chunks "
    f"(chunk_size_chars={CHUNK_SIZE_CHARS}, overlap_chars={OVERLAP_CHARS}, min_chunk_chars={MIN_CHUNK_CHARS}):"
)
for i, chunk in enumerate(_preview_chunks):
    print(f"  [{i}] ({len(chunk)} chars) {chunk[:80]!r}")

production_index_report = build_index_from_text_file(
    text_path=RAG_TEXT_KB_PATH,
    out_path=RAG_INDEX_PATH,
    embedding_model=EMBEDDING_MODEL,
    vector_db=VECTOR_DB,
    chunk_size_chars=CHUNK_SIZE_CHARS,
    overlap_chars=OVERLAP_CHARS,
    min_chunk_chars=MIN_CHUNK_CHARS,
)
print()
print(json.dumps(production_index_report, indent=2))

## 12. Verify retrieval before touching the model

Loads the index just built and runs it through a few real questions -- entirely on CPU, no GPU or
model weights involved -- so a broken embedding/index/chunking problem surfaces here with a clear
top-hit printout, rather than as a vague "the assistant hallucinated" symptom after the 7B model is
already loaded. Replace `_sample_questions` with questions relevant to your own knowledge base.

In [ ]:
if ENABLE_RAG:
    from rag.retriever import Retriever

    _verify_retriever = Retriever(embedding_model=EMBEDDING_MODEL, vector_db=VECTOR_DB)
    _verify_retriever.load_index(RAG_INDEX_PATH)

    # In-scope: should retrieve a clearly relevant chunk as the #1 hit.
    _in_scope_questions = [
        "What is RobotBulls and what does the company do?",
        "What is the Yield Bull?",
        "What blockchain does the RBT token operate on?",
        "When did RobotBulls participate in the World Economic Forum?",
    ]
    # Out-of-scope: STRICT_SCOPE's instruction wording (Section 10) is what makes the assistant
    # decline these -- not retrieval itself, which always returns its nearest chunks regardless of
    # how irrelevant the question is. Printed here so you can see the score gap (if any) for your
    # own knowledge base before deciding whether SCORE_THRESHOLD is worth setting too.
    _out_of_scope_questions = [
        "What is the capital of France?",
        "Can you give me a recipe for chocolate cake?",
    ]

    for label, questions in [("in-scope", _in_scope_questions), ("out-of-scope", _out_of_scope_questions)]:
        for question in questions:
            result = _verify_retriever.retrieve_context(question, top_k=3)
            print(f"[{label}] Q: {question}")
            for text, score in zip(result["contexts"], result["scores"]):
                print(f"   score={score:.3f}  {text[:100]!r}")
            print()
else:
    print("ENABLE_RAG is False -- skipping retrieval verification.")

## 13. Start the streaming server

Launches `moshi.server` with the RAG configuration from Section 10. When `ENABLE_RAG = True`,
every connection is grounded in the knowledge base built above via the same `<system>...<system>`
injection mechanism PersonaPlex's own persona prompt uses (Mode C), including the token-budget
guard that keeps a single injection burst from overflowing the model's attention window. No
connection ever needs to supply its own `rag_query` (the browser web UI has no way to): it falls
back to `RAG_DEFAULT_QUERY` from Section 10, then further still to the whole knowledge base (now
safely capped by `MAX_INJECTION_TOKENS`/`INJECTION_RESERVE_FRAMES` instead of being uncapped) if
even that's unset.

Safe to re-run after editing Section 10's config -- it stops whatever's currently running on
`SERVER_PORT` first.

In [ ]:
import subprocess
import sys
import time

if "server_proc" in globals() and server_proc.poll() is None:
    print(f"Stopping the currently running server (PID {server_proc.pid})...")
    server_proc.terminate()
    try:
        server_proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=30)
    print("Server stopped.")

LOG_PATH = os.path.join(WORKSPACE, "personaplex_server.log")
env = os.environ.copy()
if ENABLE_RAG:
    # rag/ lives at REPO_DIR (a sibling of moshi/), but moshi.server is launched with
    # cwd=REPO_DIR/moshi below -- put REPO_DIR on PYTHONPATH so --rag-enable can import it.
    env["PYTHONPATH"] = REPO_DIR + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

cmd = [sys.executable, "-m", "moshi.server", "--host", SERVER_HOST, "--port", str(SERVER_PORT)]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]
if ENABLE_RAG:
    cmd += [
        "--rag-enable",
        "--rag-index", RAG_INDEX_PATH,
        "--rag-top-k", str(TOP_K),
        "--rag-embedding-model", EMBEDDING_MODEL,
        "--rag-log-dir", rag_config.log_dir,
        "--rag-injection-mode", INJECTION_MODE,
        "--rag-default-query", RAG_DEFAULT_QUERY,
        "--rag-injection-reserve-frames", str(INJECTION_RESERVE_FRAMES),
        "--rag-refusal-message", REFUSAL_MESSAGE,
    ]
    if MAX_INJECTION_TOKENS is not None:
        cmd += ["--rag-max-injection-tokens", str(MAX_INJECTION_TOKENS)]
    if SCORE_THRESHOLD is not None:
        cmd += ["--rag-score-threshold", str(SCORE_THRESHOLD)]
    if not STRICT_SCOPE:
        cmd += ["--rag-no-strict-scope"]
    if VAD_ENABLED:
        cmd += ["--rag-vad-enable"]

print("Launching:", " ".join(cmd))
log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Server launched with PID {server_proc.pid}. Logs: {LOG_PATH}")

In [ ]:
def _tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
TIMEOUT_S = 900
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(_tail(LOG_PATH))
        raise RuntimeError(f"Server exited early with return code {server_proc.returncode}. See log above.")
    if READY_MARKER in open(LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(_tail(LOG_PATH))
if not ready:
    raise TimeoutError(f"Server did not report readiness within {TIMEOUT_S}s.")
print()
print("Server is up and warmed up -- open the URL printed above to talk to it.")

## 14. Stop the server (run only when you want to shut it down)

This is a management utility cell, **not** part of the linear startup flow -- running it will
terminate the backend you just verified above. Run it deliberately when you're done with the
session, not as part of a top-to-bottom "Run All".

In [ ]:
# # Intentionally NOT meant to run automatically as part of the startup sequence above.
# try:
#     server_proc.terminate()
#     server_proc.wait(timeout=15)
#     print(f"Server process {server_proc.pid} stopped.")
# except NameError:
#     print("No server_proc in scope -- nothing to stop.")
# except subprocess.TimeoutExpired:
#     server_proc.kill()
#     print(f"Server process {server_proc.pid} killed after not stopping gracefully.")